# RSNA Knee — ensemble inference (internet OFF)

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-060** | `hlly34/rsna-knee-inference` | pair blend: ConvNeXt dense 5f + Raptor repro 5f, 50/50 family rank weights (pooled OOF 0.8852) |

Scores the hidden test set with every fold model attached from the weights dataset `Kaggle Models: rsna-knee-convnext-s-288-dense, rsna-knee-raptor-repro` (uploaded by `scripts/vast/submit.sh`). Test DICOMs go through exactly the same preprocessing as training (`src/dataset/dicom.py`); each model's predictions are converted to per-finding ranks (AUC only depends on order) and averaged.

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-060", "notebook": "hlly34/rsna-knee-inference", "what": "pair blend: ConvNeXt dense 5f + Raptor repro 5f, 50/50 family rank weights (pooled OOF 0.8852)"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
FAMILY_WEIGHTS = 'convnext_small_288_dense=0.5,raptor_repro=0.5'
INFER_ARGS = []
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=7):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


## 1. Offline dependencies and checkpoints
The pylibjpeg wheels come from a training notebook's output (needed for JPEG-compressed DICOMs).

In [ ]:
WHEELS = walk_find(lambda d, f: d if any(x.startswith("pylibjpeg") and x.endswith(".whl") for x in f) else None)
if WHEELS:
    # --no-deps: never let an offline wheel (e.g. numpy) replace the image's own packages
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--find-links", WHEELS,
                    "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
CKPTS = []
walk_find(lambda d, f: CKPTS.extend(os.path.join(d, x) for x in sorted(f) if x.endswith(".pt")) and None)
print("wheels:", WHEELS)
try:
    import pylibjpeg, libjpeg, openjpeg  # noqa: F401  JPEG / JPEG-2000 DICOM decoders
    print("JPEG decoders: OK")
except ImportError as e:
    print("JPEG decoders MISSING:", e)
for c in CKPTS: print(" ", c)
assert CKPTS, "attach the training notebooks' outputs"

## 2. Predict and rank-average

In [ ]:
t0 = time.time()
from src.models.inference import main
sys.argv = ["inference", "--data_dir", COMP, "--ckpts", *CKPTS, "--rank_average", "--cache_dir", "/tmp/knee_cache",
            *(["--family_weights", FAMILY_WEIGHTS] if FAMILY_WEIGHTS else []), *INFER_ARGS,
            "--workers", str(os.cpu_count()), "--out", "/kaggle/working/submission.csv"]
main()
print(f"inference {time.time() - t0:.0f}s")

In [ ]:
import numpy as np, pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
files = glob.glob("/tmp/knee_cache/*/*.npz")   # only written in --mode cache (stream keeps stacks in memory)
EXP.update(models=len(CKPTS), test_studies=len(sub), seconds=round(time.time() - t0),
           ckpts=[os.path.join(os.path.basename(os.path.dirname(c)), os.path.basename(c)) for c in CKPTS],
           ckpt_paths=CKPTS, family_weights=FAMILY_WEIGHTS,
           upload_ids=[])
walk_find(lambda d, f: EXP["upload_ids"].extend(sorted(x for x in f if x.startswith("UPLOAD_"))) and None)
if files:
    EXP["test_slot_coverage"] = round(float(np.mean([np.load(f)["valid"].mean() for f in files])), 4)
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1)
print(EXP); sub.head()